# Sentinel offline enrichment pipeline

Runs the GPU-bound part of Sentinel's video pipeline: starts a local
[DAM](https://github.com/NVlabs/describe-anything) server, then runs
`pipeline/run_pipeline.py` against a demo clip to populate real
segments + embeddings in Postgres.

**Before running:**
1. `Runtime -> Change runtime type -> T4 GPU` (a free-tier GPU is enough).
2. Click the key icon in the left sidebar (Secrets) and add two secrets:
   - `DATABASE_URL` - the Neon Postgres connection string
   - `LLM_API_KEY` - the Gemini API key

   Toggle "Notebook access" on for both.
3. Run the cells in order. The DAM-3B-Video checkpoint is several GB, so
   the model-loading cell can take a few minutes.

If a cell fails, the error message is usually enough to fix - paste it
back to continue debugging.

In [ ]:
!nvidia-smi

In [ ]:
!apt-get -qq update && apt-get -qq install -y ffmpeg
!test -d /content/sentinel && git -C /content/sentinel pull || git clone https://github.com/Mohit-Kirtane/sentinel.git /content/sentinel
%cd /content/sentinel

In [ ]:
!test -d /content/describe-anything || git clone https://github.com/NVlabs/describe-anything.git /content/describe-anything
%cd /content/describe-anything
!pip install -q -v .
# DAM's pyproject.toml only pins transformers>=4.41.2 (no upper bound), so it
# pulls in whatever Colab has preinstalled. transformers 5.x removed
# no_init_weights from modeling_utils, which DAM's code imports directly -
# force a compatible 4.x release.
!pip install -q "transformers<5"

In [ ]:
%cd /content/sentinel
!pip install -q -r pipeline/requirements.txt

In [ ]:
import os
import subprocess

server_log = open("/content/dam_server.log", "w")
server_proc = subprocess.Popen(
    [
        "python", "dam_server.py",
        "--model-path", "nvidia/DAM-3B-Video",
        "--conv-mode", "v1",
        "--prompt-mode", "focal_prompt",
        "--temperature", "0.2",
        "--top_p", "0.9",
        "--num_beams", "1",
        "--max_new_tokens", "512",
        "--workers", "1",
        "--image_video_joint_checkpoint",
    ],
    cwd="/content/describe-anything",
    stdout=server_log,
    stderr=subprocess.STDOUT,
)
print(f"DAM server starting, pid={server_proc.pid}")

In [ ]:
import socket
import time


def wait_for_port(port, host="localhost", timeout=1200):
    start = time.time()
    while time.time() - start < timeout:
        if server_proc.poll() is not None:
            raise RuntimeError(
                "DAM server process exited early - check /content/dam_server.log"
            )
        try:
            with socket.create_connection((host, port), timeout=2):
                return True
        except OSError:
            time.sleep(5)
    return False


if wait_for_port(8000):
    print("DAM server is up on port 8000")
else:
    print("Timed out waiting for the DAM server - check the log:")
    !tail -n 60 /content/dam_server.log

In [ ]:
from google.colab import userdata

os.environ["DATABASE_URL"] = userdata.get("DATABASE_URL")
os.environ["LLM_API_KEY"] = userdata.get("LLM_API_KEY")
os.environ["PYTHONPATH"] = "/content/sentinel/backend"
%cd /content/sentinel

In [ ]:
!python -m pipeline.run_pipeline \
  --video frontend/public/demo-videos/street-intersection.mp4 \
  --video-id street-intersection \
  --title "Street Intersection Camera" \
  --dam-server-url http://localhost:8000 \
  --interval-seconds 1.5 \
  --frame-skip 2

## Verify

Confirms real segments landed in Postgres for this video.

In [ ]:
import sys

sys.path.insert(0, "/content/sentinel/backend")

from sqlalchemy import create_engine, select
from sqlalchemy.orm import sessionmaker

from app.db.models import Segment

engine = create_engine(os.environ["DATABASE_URL"])
with sessionmaker(bind=engine)() as session:
    segments = session.execute(
        select(Segment).where(Segment.video_id == "street-intersection")
    ).scalars().all()

print(f"{len(segments)} segments stored")
for s in segments:
    print(f"[{s.start_ts}-{s.end_ts}s] {s.description}")